In [ ]:
COLABNOMAD_REPO = 'https://github.com/pedro-labsabs/ColabNomad.git'
COLABNOMAD_REF = 'v0.1.0'
COLABNOMAD_RELEASE = 'v0.1.0'
TARGET_REPO = ''  # user must set this
TARGET_REF = ''


In [ ]:
from pathlib import Path
import os
import subprocess
from google.colab import userdata

if not TARGET_REPO.strip():
    raise ValueError('TARGET_REPO must be set')

kit = Path('/content/colabnomad-kit')
if not kit.exists():
    subprocess.run(['git', 'clone', '--no-checkout', COLABNOMAD_REPO, str(kit)], check=True)
subprocess.run(['git', '-C', str(kit), 'remote', 'set-url', 'origin', COLABNOMAD_REPO], check=True)

subprocess.run(['git', '-C', str(kit), 'fetch', '--depth=1', 'origin', COLABNOMAD_REF], check=True)
subprocess.run(['git', '-C', str(kit), 'checkout', '--detach', 'FETCH_HEAD'], check=True)

bootstrap_env = os.environ.copy()
for secret_name in ('GITHUB_TOKEN', 'OPENCODE_API_KEY'):
    try:
        secret_value = userdata.get(secret_name)
    except (KeyError, RuntimeError, AttributeError):
        secret_value = None
    if secret_value:
        bootstrap_env[secret_name] = secret_value

bootstrap = ['python', str(kit / 'colab' / 'bootstrap.py'), '--target-repo', TARGET_REPO, '--release', COLABNOMAD_RELEASE]
if TARGET_REF:
    bootstrap.extend(['--target-ref', TARGET_REF])
subprocess.run(bootstrap, check=True, env=bootstrap_env)
